# Step 4: Train bge-base on Mistral-generated Pairs

**Goal:** Test whether evaluation contamination is generator-specific.

**Hypothesis:** A model trained on mistral-generated pairs should score
lower on gemma-generated synthetic eval (different generator = less
contamination) but similar on manual eval (human queries are unaffected
by which LLM generated the training data).

**What changes vs step 2:**
- Training data: `training_pairs_mistral.json` instead of `training_pairs.json`
- Validation: `manual_val.json` (10 human queries) instead of synthetic pairs
- Output: `experiments/bge_mistral/`

**Everything else identical:** bge-base-en-v1.5, MNRL, 2700 pairs,
5 epochs, batch 32, warmup 100, seed 42.

## 0. Runtime check

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip())
if 'T4' not in result.stdout:
    print('WARNING: Not on T4. Go to Runtime > Change runtime type > T4 GPU')
else:
    print('Good to go.')

## 1. Install dependencies

In [ ]:
%%capture
!pip install sentence-transformers faiss-cpu accelerate

## 2. Upload files

Upload these files one at a time when the dialog appears:
- `data/training_pairs_mistral.json`
- `data/manual_val.json`
- `experiments/baseline/baseline_chunks.pkl`

In [ ]:
from google.colab import files
import os

os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/experiments/baseline', exist_ok=True)
os.makedirs('/content/experiments/bge_mistral/model', exist_ok=True)

dest_map = {
    'training_pairs_mistral.json': '/content/data/training_pairs_mistral.json',
    'manual_val.json':             '/content/data/manual_val.json',
    'baseline_chunks.pkl':         '/content/experiments/baseline/baseline_chunks.pkl',
}

print('Upload one file at a time — run this cell 3 times')
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = dest_map.get(fname, f'/content/{fname}')
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'✓ {fname} -> {dest}')

print('\nUploaded so far:')
for path in dest_map.values():
    status = '✓' if os.path.exists(path) else '✗ missing'
    print(f'  {status}  {path}')

## 3. Config

In [ ]:
import random
import numpy as np

BASE_MODEL   = 'BAAI/bge-base-en-v1.5'
TRAIN_SIZE   = 2700
EPOCHS       = 5
BATCH_SIZE   = 32
WARMUP_STEPS = 100
SEED         = 42
OUT_DIR      = '/content/experiments/bge_mistral/model'

random.seed(SEED)
np.random.seed(SEED)

print(f'Base model   : {BASE_MODEL}')
print(f'Training data: training_pairs_mistral.json')
print(f'Validation   : manual_val.json (10 human queries)')
print(f'Train size   : {TRAIN_SIZE}')
print(f'Epochs       : {EPOCHS}')

## 4. Load data

In [ ]:
import json

# Mistral-generated training pairs
with open('/content/data/training_pairs_mistral.json') as f:
    all_mistral = json.load(f)

# Same split logic as all previous runs
# last 100 = held-out eval, never touched
train_pairs = all_mistral[:TRAIN_SIZE]

# Human val queries (step 9)
with open('/content/data/manual_val.json') as f:
    val_pairs = json.load(f)

print(f'Mistral pairs total : {len(all_mistral)}')
print(f'Training pairs      : {len(train_pairs)}')
print(f'Validation queries  : {len(val_pairs)} (human-written)')
print(f'\nExample training pair:')
print(f'  Query    : {train_pairs[0]["query"][:80]}')
print(f'  Positive : {train_pairs[0]["positive"][:80]}')

## 5. Build validation evaluator (human queries)

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

queries  = {str(i): p['query']    for i, p in enumerate(val_pairs)}
corpus   = {str(i): p['positive'] for i, p in enumerate(val_pairs)}
relevant = {str(i): {str(i)}      for i in range(len(val_pairs))}

evaluator = InformationRetrievalEvaluator(
    queries, corpus, relevant, name='val-human-mistral'
)
print(f'Human validation evaluator ready — {len(val_pairs)} queries')

## 6. Train (~10 min on T4)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader

examples = [InputExample(texts=[p['query'], p['positive']]) for p in train_pairs]
model    = SentenceTransformer(BASE_MODEL)
loader   = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
loss     = MultipleNegativesRankingLoss(model)

print(f'Training — {len(examples)} pairs, {EPOCHS} epochs')

model.fit(
    train_objectives=[(loader, loss)],
    evaluator=evaluator,
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    evaluation_steps=len(loader),
    output_path=OUT_DIR,
    save_best_model=True,
    show_progress_bar=True,
    use_amp=True
)

print(f'\nSaved to {OUT_DIR}')

## 7. Build FAISS index (~3 min)

In [ ]:
import pickle
import faiss

with open('/content/experiments/baseline/baseline_chunks.pkl', 'rb') as f:
    chunks = pickle.load(f)
print(f'Loaded {len(chunks)} chunks')

model      = SentenceTransformer(OUT_DIR)
texts      = [c['text'] for c in chunks]
embeddings = model.encode(texts, batch_size=256, show_progress_bar=True,
                           normalize_embeddings=True)

dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

idx_path = '/content/experiments/bge_mistral/bge_mistral.faiss'
faiss.write_index(index, idx_path)
print(f'Saved: {index.ntotal} vectors, dim={dim}')

## 8. Download

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/bge_mistral', 'zip',
                    '/content/experiments', 'bge_mistral')
files.download('/content/bge_mistral.zip')
print('Unzip into your experiments/ folder on Mac.')
print('Then add bge_mistral to BGE_RUNS in step4d_eval_bge.py and run:')
print('  python step4d_eval_bge.py --eval manual')
print('  python step4d_eval_bge.py --eval synthetic')